## **DESCRIPCIÓN DEL RETO**
### Presenta:  Jesús Carlos Sánchez Guzmán
### **Contexto**
En una universidad, miles de alumnos hacen cada semestre las mismas preguntas a Servicios Escolares: ¿hasta cuándo puedo reinscribirme?, ¿cómo doy de baja una materia?, ¿qué requisitos pide el servicio social?, ¿qué promedio necesito para una beca? Las respuestas existen, pero están dispersas en reglamentos, calendarios y convocatorias en PDF que casi nadie lee completos.

### **Problema**
- El personal administrativo se satura respondiendo preguntas repetitivas.
- Los alumnos esperan horas o días por una respuesta que ya está documentada.
- Un modelo de lenguaje genérico **no conoce los reglamentos de la universidad** y puede inventar fechas, requisitos o costos, con consecuencias reales: un trámite perdido, un recargo o una beca no solicitada a tiempo.

### **Reto**
Construir un **asistente de trámites y reglamentos universitarios** que responda las dudas de los alumnos de forma inmediata, correcta y verificable.

### **Objetivos**
1. Comparar estrategias de prompting (zero-shot, few-shot y chain-of-thought) y reconocer sus límites.
2. Demostrar que un prompt, por sí solo, no da al modelo información que nunca vio.
3. Implementar un sistema **RAG** que recupere los fragmentos relevantes de los documentos oficiales mediante *embeddings* y similitud semántica.
4. Generar respuestas **fundamentadas únicamente en esos documentos** y que **citen su fuente**.
5. Evitar alucinaciones: si la información no está en la base de conocimiento, el asistente debe decirlo y derivar al alumno a Servicios Escolares.

### **Criterios de éxito**
| Criterio | Qué se espera |
|---|---|
| **Precisión** | La respuesta coincide con lo que dice el documento oficial. |
| **Trazabilidad** | Cada respuesta indica la fuente consultada (por ejemplo, "Reglamento académico, art. 23"). |
| **Honestidad** | Ante una pregunta fuera de alcance, no inventa y deriva al alumno. |
| **Mejora demostrable** | La comparación con y sin RAG muestra una respuesta más confiable con RAG. |

### **Herramientas**
- **Groq** (`openai/gpt-oss-20b`) como modelo generativo.
- **sentence-transformers** (`paraphrase-multilingual-MiniLM-L12-v2`) para los embeddings.
- **Google Colab** como entorno de ejecución.

### **Alcance y supuestos**
La base de conocimiento de este notebook usa datos de ejemplo de una universidad ficticia. En un caso real se sustituye por los documentos oficiales de la institución.

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [4]:
# Instalar cliente de Groq y leer la API key (Colab Secrets, variable de entorno o captura manual)
!pip install -q groq

import os
from getpass import getpass
from groq import Groq

try:
    # Si estás en Google Colab, usa el panel de Secrets
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except ImportError:
    # En local: usa la variable de entorno o, si no existe, pídela sin mostrarla en pantalla
    api_key = os.environ.get("GROQ_API_KEY") or getpass("Pega tu GROQ_API_KEY: ")

client = Groq(api_key=api_key)
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [5]:
# Prompt de clasificación en modo zero-shot

prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero_shot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt_zero_shot}
    ]
)
#print(response_zero_shot)
print('Zero-shot: ' + response_zero_shot.choices[0].message.content)



Zero-shot: Mixto.


In [6]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.
Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo
Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo
Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few_shot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": prompt_few_shot}
    ]
)
#print(response_few_shot)
print('few-shot: ' + response_few_shot.choices[0].message.content)

few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [7]:
# Razonamiento paso a paso (chain-of-thought)
problem = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)
response_cot = client.chat.completions.create(
    model = "openai/gpt-oss-20b",
    messages = [{
        'role': 'user',
        'content': problem}]
)
print(response_cot.choices[0].message.content)

**Paso a paso:**

1. **Define las variables**  
   - Velocidad del primer tren (V₁): \(80 \text{ km/h}\)  
   - Velocidad del segundo tren (V₂): \(120 \text{ km/h}\)  
   - Tiempo que el primer tren viaja antes de que el segundo salga: \(2 \text{ h}\)

2. **Calcula la distancia adelantada por el primer tren**  
   \[
   \text{Distancia adelantada} = V₁ \times 2 \text{ h} = 80 \, \text{km/h} \times 2 \text{ h} = 160 \text{ km}
   \]

3. **Determina la velocidad relativa**  
   El segundo tren avanza más rápido que el primero, por lo que la velocidad con la que se acorta la brecha es la diferencia de sus velocidades:
   \[
   V_{\text{relativa}} = V₂ - V₁ = 120 \text{ km/h} - 80 \text{ km/h} = 40 \text{ km/h}
   \]

4. **Calcula el tiempo necesario para alcanzar al primer tren**  
   \[
   \text{Tiempo de persecución} = \frac{\text{Distancia adelantada}}{V_{\text{relativa}}}
   = \frac{160 \text{ km}}{40 \text{ km/h}} = 4 \text{ h}
   \]

5. **Resultado**  
   El segundo tren tardará **4

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [8]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)
response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)
print(response_alucinacion.choices[0].message.content)

Lo siento, no dispongo de esa información.


## **RAG: ASISTENTE DE TRÁMITES Y REGLAMENTOS UNIVERSITARIOS**

**Caso de uso:** los alumnos hacen constantemente las mismas preguntas a Servicios Escolares (fechas de reinscripción, bajas, servicio social, becas, titulación, etc.). Un modelo de lenguaje genérico no conoce los reglamentos ni los calendarios de *tu* universidad y puede inventar fechas o requisitos, con consecuencias reales para el alumno.

RAG resuelve esto en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de los **documentos oficiales** (usando *embeddings* y similitud semántica); después, esos fragmentos se entregan al modelo junto con la pregunta para generar una respuesta fundamentada que **cita su fuente**.

In [9]:
# Instalar sentence-transformers
!pip install -q sentence-transformers

In [10]:
# Definir la base de conocimiento universitaria (trámites y reglamentos) y generar sus embeddings
from sentence_transformers import SentenceTransformer, util

# Modelo multilingüe (funciona bien con español)
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Cada fragmento guarda su fuente para poder citarla en la respuesta.
# NOTA: son datos de ejemplo (universidad ficticia). En un caso real se cargan desde los PDF/documentos oficiales.
base_conocimiento = [
    {"fuente": "Calendario escolar, sec. Reinscripción",
     "texto": "La reinscripción al semestre se realiza del 3 al 14 de enero a través del portal de alumnos. Después de esa fecha se aplica un recargo del 10% por extemporaneidad."},
    {"fuente": "Calendario escolar, sec. Bajas de materias",
     "texto": "Las bajas de materias pueden solicitarse hasta la semana 8 del semestre, mediante el formato de baja firmado por el coordinador de carrera. Una materia dada de baja no cuenta como intento reprobado."},
    {"fuente": "Reglamento académico, art. 23",
     "texto": "Para aprobar una materia el alumno debe obtener una calificación mínima de 70 y cumplir con al menos el 80% de asistencia a clases."},
    {"fuente": "Reglamento académico, art. 31",
     "texto": "Un alumno puede presentar examen extraordinario hasta en dos ocasiones por materia. Cada solicitud debe pagarse y registrarse con al menos 5 días hábiles de anticipación."},
    {"fuente": "Reglamento de servicio social, art. 4",
     "texto": "Para iniciar el servicio social el alumno debe haber cubierto al menos el 70% de los créditos de su plan de estudios. El servicio social comprende 480 horas que se cubren en un periodo de entre 6 y 12 meses."},
    {"fuente": "Reglamento de servicio social, art. 9",
     "texto": "Para liberar el servicio social se debe entregar la carta de terminación firmada por la institución receptora y los reportes mensuales de actividades."},
    {"fuente": "Convocatoria de becas 2026, requisitos",
     "texto": "La beca de excelencia académica se otorga a alumnos con promedio mínimo de 90, sin materias reprobadas ni extraordinarios, y que estén inscritos en la carga completa del semestre."},
    {"fuente": "Convocatoria de becas 2026, fechas",
     "texto": "La solicitud de beca de excelencia se realiza en línea del 15 al 30 de agosto. Se debe adjuntar la boleta de calificaciones y comprobante de domicilio."},
    {"fuente": "Guía de titulación, opciones",
     "texto": "Las opciones de titulación son tesis, examen general de egreso, estancia profesional o promedio de excelencia (mínimo 90). Es requisito haber liberado el servicio social y acreditado el nivel B1 de inglés."},
    {"fuente": "Guía de constancias",
     "texto": "Las constancias de estudios se solicitan en el portal de alumnos, tienen un costo de 50 pesos y se entregan en formato digital en un máximo de 3 días hábiles."},
    {"fuente": "Reglamento de biblioteca, art. 5",
     "texto": "Los alumnos pueden solicitar hasta 3 libros en préstamo por 7 días. La renovación puede hacerse hasta dos veces si el libro no tiene lista de espera. Cada día de retraso genera una multa de 10 pesos."},
    {"fuente": "Guía de movilidad estudiantil",
     "texto": "Para participar en intercambio académico se requiere un promedio mínimo de 85, haber cursado al menos 4 semestres y presentar un nivel B2 de inglés o equivalente."},
]

textos = [f["texto"] for f in base_conocimiento]
embeddings_base = embedder.encode(textos, convert_to_tensor=True)
print(f"Base de conocimiento: {len(base_conocimiento)} fragmentos | embeddings: {tuple(embeddings_base.shape)}")

W1011 23:10:29.126000 34240 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Base de conocimiento: 12 fragmentos | embeddings: (12, 384)


In [11]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese los más relevantes
UMBRAL = 0.30  # por debajo de esta similitud se considera que la base NO tiene la información

def recuperar_fragmentos(pregunta, top_k=3, umbral=UMBRAL):
    """Devuelve hasta top_k fragmentos (fuente, texto, score) con similitud coseno >= umbral."""
    emb_pregunta = embedder.encode(pregunta, convert_to_tensor=True)
    similitudes = util.cos_sim(emb_pregunta, embeddings_base)[0]
    top = similitudes.topk(min(top_k, len(base_conocimiento)))
    resultados = []
    for score, i in zip(top.values, top.indices):
        if float(score) >= umbral:
            f = base_conocimiento[int(i)]
            resultados.append({"fuente": f["fuente"], "texto": f["texto"], "score": float(score)})
    return resultados

pregunta = "¿Hasta cuándo puedo dar de baja una materia?"
for r in recuperar_fragmentos(pregunta):
    print(f"[{r['score']:.3f}] ({r['fuente']}) {r['texto']}")

[0.372] (Calendario escolar, sec. Bajas de materias) Las bajas de materias pueden solicitarse hasta la semana 8 del semestre, mediante el formato de baja firmado por el coordinador de carrera. Una materia dada de baja no cuenta como intento reprobado.
[0.300] (Reglamento de biblioteca, art. 5) Los alumnos pueden solicitar hasta 3 libros en préstamo por 7 días. La renovación puede hacerse hasta dos veces si el libro no tiene lista de espera. Cada día de retraso genera una multa de 10 pesos.


In [12]:
# Enviar la pregunta junto con los fragmentos recuperados al modelo y mostrar la respuesta con RAG (citando fuentes)
def responder_con_rag(pregunta, top_k=3):
    fragmentos = recuperar_fragmentos(pregunta, top_k=top_k)

    # Si no hay fragmentos suficientemente similares, no se llama al modelo: se evita alucinar.
    if not fragmentos:
        return "No encontré esa información en los documentos oficiales. Te recomiendo acudir a Servicios Escolares.", []

    contexto = "\n".join(f"[{n}] ({f['fuente']}) {f['texto']}" for n, f in enumerate(fragmentos, 1))

    prompt_rag = f"""Eres el asistente de trámites y reglamentos de la universidad.
Responde la pregunta del alumno usando ÚNICAMENTE el contexto proporcionado.
Si la respuesta no está en el contexto, di que no tienes esa información y sugiere acudir a Servicios Escolares.
Responde en español, de forma breve y clara, e indica al final la fuente usada entre paréntesis.

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:"""

    respuesta = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt_rag}]
    )
    return respuesta.choices[0].message.content, fragmentos

pregunta = "¿Hasta cuándo puedo dar de baja una materia?"
respuesta, fuentes = responder_con_rag(pregunta)
print("Pregunta:", pregunta)
print("Respuesta con RAG:", respuesta)
print("Fuentes consultadas:", [f["fuente"] for f in fuentes])

Pregunta: ¿Hasta cuándo puedo dar de baja una materia?
Respuesta con RAG: Puedes dar de baja una materia hasta la semana 8 del semestre, mediante el formato de baja firmado por el coordinador de carrera. (Calendario escolar, sec. Bajas de materias)
Fuentes consultadas: ['Calendario escolar, sec. Bajas de materias', 'Reglamento de biblioteca, art. 5']


### **PRUEBAS DEL ASISTENTE**

Se prueban preguntas típicas de alumnos y una pregunta fuera de la base de conocimiento, para verificar que el asistente cita sus fuentes y no inventa respuestas.

In [13]:
# Probar el asistente con varias preguntas típicas de alumnos (incluida una fuera de la base de conocimiento)
preguntas_prueba = [
    "¿Cuántas horas dura el servicio social y cuándo puedo empezarlo?",
    "¿Qué promedio necesito para la beca de excelencia?",
    "¿Cuánto cuesta una constancia de estudios y cuánto tarda?",
    "¿Qué pasa si entrego tarde un libro de la biblioteca?",
    "¿Cuál es el menú de la cafetería hoy?",   # fuera de la base: el asistente no debe inventar
]

for p in preguntas_prueba:
    respuesta, fuentes = responder_con_rag(p)
    print("Pregunta:", p)
    print("Respuesta:", respuesta)
    print("Fuentes:", [f["fuente"] for f in fuentes] or "—")
    print("-" * 80)

Pregunta: ¿Cuántas horas dura el servicio social y cuándo puedo empezarlo?
Respuesta: El servicio social tiene una duración de **480 horas** y puede iniciarse una vez que el alumno haya cubierto **al menos el 70 % de los créditos** de su plan de estudios. (Reglamento de servicio social, art. 4)
Fuentes: ['Reglamento de servicio social, art. 4', 'Reglamento de servicio social, art. 9', 'Calendario escolar, sec. Reinscripción']
--------------------------------------------------------------------------------
Pregunta: ¿Qué promedio necesito para la beca de excelencia?
Respuesta: Necesitas un promedio mínimo de 90. (Convocatoria de becas 2026, requisitos)
Fuentes: ['Guía de titulación, opciones', 'Guía de movilidad estudiantil', 'Convocatoria de becas 2026, requisitos']
--------------------------------------------------------------------------------
Pregunta: ¿Cuánto cuesta una constancia de estudios y cuánto tarda?
Respuesta: Cuesta 50 pesos y se entrega en formato digital en un máximo de

### **CON RAG VS. SIN RAG**

La misma pregunta, con y sin acceso a los documentos oficiales.

In [14]:
# Comparación: la misma pregunta con y sin RAG
pregunta = "¿Qué promedio necesito para la beca de excelencia de mi universidad?"

sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": pregunta + " Respuesta muy breve y corta."}]
)
con_rag, fuentes = responder_con_rag(pregunta)

print("SIN RAG:", sin_rag.choices[0].message.content)
print("\nCON RAG:", con_rag)
print("Fuentes:", [f["fuente"] for f in fuentes])

SIN RAG: Generalmente se requiere un promedio de **9.0 sobre 10** (o el 90 % si la escala es porcentual).

CON RAG: Necesitas un promedio mínimo de 90. (Convocatoria de becas 2026)
Fuentes: ['Guía de movilidad estudiantil', 'Guía de titulación, opciones', 'Convocatoria de becas 2026, requisitos']


### **RESUMEN DE LA SOLUCIÓN**

**Reto:** reducir la carga de Servicios Escolares y dar a los alumnos respuestas inmediatas, correctas y verificables sobre trámites y reglamentos.

**Solución:** un asistente basado en RAG sobre documentos oficiales de la universidad.

1. **Prompting (secciones previas):** zero-shot y few-shot para controlar el formato de salida, chain-of-thought para razonar paso a paso, y la demostración de que un prompt por sí solo no da información que el modelo nunca vio.
2. **Base de conocimiento:** reglamento académico, calendario escolar, servicio social, becas, titulación, constancias, biblioteca y movilidad, divididos en fragmentos con su **fuente** (datos de ejemplo de una universidad ficticia).
3. **Recuperación:** cada fragmento se convierte en un *embedding* con `sentence-transformers` (`paraphrase-multilingual-MiniLM-L12-v2`) y, para cada pregunta, se recuperan los `top_k` más similares por **similitud coseno**.
4. **Control de alucinaciones:**
   - Un **umbral de similitud** evita llamar al modelo cuando la base no contiene la respuesta.
   - El prompt obliga al modelo a responder **solo con el contexto** y a **citar la fuente**.
5. **Evaluación:** preguntas típicas de alumnos, una pregunta fuera de alcance y una comparación con/sin RAG.

**Siguientes pasos para un despliegue real:** cargar los PDF oficiales (con *chunking*), exponer el asistente en un chat web o WhatsApp, registrar las preguntas sin respuesta para detectar vacíos en la documentación y actualizar la base cada semestre.

**Conclusión:** el prompt engineering mejora *cómo* responde el modelo; RAG mejora *con qué información* responde, y eso es lo que hace confiable al asistente en un contexto institucional.